# 11 — Amari's Bridge

The identity the whole course has been walking toward: the Sinkhorn plan is the **KL projection** of the Gibbs kernel onto the transportation polytope. Transport geometry *is* information geometry.

**Prerequisites:** `08_kantorovich_duality`, `09_entropic_regularization`, `10_sinkhorn_algorithm`, and `02/02_kl_divergence`, `02/06_fisher_rao_metric`.

**What you'll be able to do**
- State Amari's identity: the entropic OT plan is the KL projection of the Gibbs kernel onto $T(a, b)$.
- Verify the projection numerically — every marginal-preserving perturbation increases the KL.
- Read how this closes the M2 (information) ↔ M3 (transport) loop and seeds quantum Sinkhorn.

You have a fast algorithm (10). Here is the payoff: a single identity that explains *why* it works, and reveals that the transport geometry you have been building is the information geometry of Module 2 wearing a different hat. This is the bridge the course has been heading toward — and the one that survives the lift to quantum states.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost
from qot_course.transport.sinkhorn import gibbs_kernel, sinkhorn, kl_to_kernel

np.random.seed(42)
viz.use_course_style()

## Where we are

Three bricks brought us here: `08_kantorovich_duality` gave the dual face and its potentials, `09_entropic_regularization` added the entropy bonus and the Gibbs kernel $K = e^{-C/\varepsilon}$, and `10_sinkhorn_algorithm` solved it in five lines. Now we ask *why* that iteration is the right thing to do — and the answer reaches back to the KL divergence of Module 2.

## 1. Sinkhorn is a KL projection

Plug the Gibbs kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ into the KL divergence and expand:

$$ \varepsilon\, \mathrm{KL}(P \,\|\, K) = \sum_{ij} P_{ij}\bigl(\varepsilon \log P_{ij} + C_{ij}\bigr) + \mathrm{const} = \langle C, P\rangle - \varepsilon\, H(P) + \mathrm{const}. $$

The right-hand side is exactly the entropic OT objective. So **minimising the entropic cost over $T(a, b)$ is the same as minimising $\mathrm{KL}(P \,\|\, K)$ over $T(a, b)$** — the Sinkhorn plan is the **KL projection of the Gibbs kernel onto the transportation polytope** (Amari, 2016). The iteration itself is alternating Bregman projections under KL: each half-step is the multiplicative row/column rescaling you already know.

If $P^\star_\varepsilon$ truly is that projection, then *any* marginal-preserving move away from it must increase the KL. Let's check.

In [ ]:
eps = 0.4
positions = np.linspace(0.0, 9.0, 16)

def bump(center, sigma=1.2):
    p = np.exp(-0.5 * ((positions - center) / sigma) ** 2)
    return p / p.sum()

src = bump(2.0)
tgt = bump(7.0)
C_demo = squared_euclidean_cost(positions, positions)

K = gibbs_kernel(C_demo, eps)
plan_star = sinkhorn(src, tgt, C_demo, epsilon=eps, n_iter=5000, tol=1e-14)
kl_star = kl_to_kernel(plan_star, K)

# Marginal-preserving 2x2-cycle perturbations: add delta*[[+1,-1],[-1,+1]] on a sub-block.
rng = np.random.default_rng(7)
perturbed_kls = []
while len(perturbed_kls) < 6:
    i1, i2 = rng.choice(plan_star.shape[0], 2, replace=False)
    j1, j2 = rng.choice(plan_star.shape[1], 2, replace=False)
    delta = 0.5 * min(plan_star[i1, j2], plan_star[i2, j1])
    if delta < 1e-12:
        continue
    P = plan_star.copy()
    P[i1, j1] += delta; P[i2, j2] += delta
    P[i1, j2] -= delta; P[i2, j1] -= delta
    if np.all(P > 0):
        perturbed_kls.append(kl_to_kernel(P, K))

print(f"KL(P*_eps || K) at the Sinkhorn fixed point   = {kl_star:.6f}")
print(f"min KL among {len(perturbed_kls)} marginal-preserving moves = {min(perturbed_kls):.6f}")
print(f"every perturbation increases the KL?  {all(k > kl_star for k in perturbed_kls)}")

In [ ]:
increases = [k - kl_star for k in perturbed_kls]

fig, ax = plt.subplots(figsize=(9, 4.2))
heights = [0.0] + increases
colors = [viz.COLORS["highlight"]] + [viz.SOURCE_COLOR] * len(increases)
labels = ["Sinkhorn\nP*"] + [f"move {i + 1}" for i in range(len(increases))]
ax.bar(range(len(heights)), heights, color=colors)
ax.set_xticks(range(len(heights)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("KL(P || K)  -  KL(P* || K)")
ax.set_title("The Sinkhorn plan sits at the KL floor: every marginal-preserving move rises", pad=12)
plt.show()

**Read the output and figure.** The Sinkhorn fixed point has the smallest $\mathrm{KL}(P \,\|\, K)$ of any coupling with the right marginals: every perturbation we tried sits strictly above it (all bars positive, $P^\star$ at the floor). So $P^\star_\varepsilon$ *is* the KL projection of the Gibbs kernel onto $T(a, b)$ — **Amari's bridge holds numerically**.

Why this is more than a reformulation:
- **It generalises.** Replace KL by any Bregman divergence and the same projection logic applies. In Module 4 the *Umegaki relative entropy* replaces the classical KL and gives **quantum Sinkhorn**.
- **It explains the speed.** Alternating Bregman projections onto affine constraints converge linearly under KL — the geometric rate of `10`.
- **It unifies the course.** The transport geometry of M3 and the information geometry of M2 are the same geometry, approached from two sides.

## 2. Dictionary update — closing the M2 ↔ M3 loop

The Sinkhorn arc adds the duality and entropic rows, and the last one is the bridge itself: the classical KL projection becomes the quantum Umegaki projection in Module 4.

| Classical | Quantum |
|-----------|---------|
| probability vector $a$ | density matrix $\rho$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ |
| Wasserstein-$p$ distance $W_p$ | quantum Wasserstein  *(Module 4)* |
| Kantorovich LP | quantum OT SDP  *(Module 4)* |
| **Kantorovich potentials $\varphi, \psi$** | **quantum potentials (SDP duals)**  *(Module 4)* |
| **Gibbs kernel $K = e^{-C/\varepsilon}$** | **matrix-exponential kernel $e^{-C/\varepsilon}$**  *(Module 4)* |
| **Sinkhorn iterations (matrix scaling)** | **quantum Sinkhorn (matrix exp + partial-trace fixed point)**  *(Module 4)* |
| **Sinkhorn = KL projection (Amari)** | **quantum Sinkhorn = Umegaki-relative-entropy projection**  *(Module 4)* |

M2 (information geometry) and M3 (transport geometry) meet here, and the same picture lifts to quantum states.

## Your turn

1. **Do the algebra.** Expand $\varepsilon\,\mathrm{KL}(P \,\|\, K)$ with $K = e^{-C/\varepsilon}$ and confirm it equals $\langle C, P\rangle - \varepsilon H(P) + \mathrm{const}$. What exactly is the constant, and why does it not affect the minimiser over $T(a, b)$?
2. **Projections, half-step by half-step.** Run a few Sinkhorn iterations and check that each half-step exactly fixes one marginal (a row rescale makes $P\mathbf{1} = a$; a column rescale makes $P^\top\mathbf{1} = b$) — alternating Bregman projections in action.
3. **Toward the quantum bridge.** Which Module-2 quantity is the quantum analogue of the KL divergence that will define quantum Sinkhorn? Find where you met it (hint: `02/09`) and state the projection it will solve.

## What you built

- You stated and proved-by-expansion Amari's identity: entropic OT minimises $\mathrm{KL}(P\|K)$ over $T(a, b)$.
- You verified the projection numerically — every marginal-preserving move increases the KL, with $P^\star$ at the floor.
- You closed the M2 ↔ M3 loop in the dictionary and saw how the bridge seeds quantum Sinkhorn.

This is one of the deepest ideas in the course: the transport geometry of Module 3 and the information geometry of Module 2 are a single geometry seen from two angles. From here, quantum optimal transport is one more change of object — swap probability vectors for density matrices, and KL for Umegaki.

## What's next

The last arc of Module 3 lifts transport to a continuous family where everything is closed-form. In `12_bures_wasserstein` we compute $W_2$ between Gaussians without any LP — and that formula is the literal doorway into quantum optimal transport.

## References

- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016), sec. 7.5.
- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *NeurIPS* (2013).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 4–5. DOI:10.1561/2200000073
- R. Sinkhorn, "A relationship between arbitrary positive matrices and doubly stochastic matrices", *Ann. Math. Statist.* 35, 876–879 (1964).

**Previous:** `notebooks/03_ClassicalOptimalTransport/10_sinkhorn_algorithm.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/12_bures_wasserstein.ipynb`